# Process and visualize the dust datasets

This notebook processes and visualizes the dust flux datasets, corresponding to empirical and simulated dust flux depositions for the Holocene and Last Glacial Maximum periods.

## Preliminaries

Import the necessary libraries and specify the data folders.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.stats import norm, lognorm
import xarray as xr
from sklearn.preprocessing import StandardScaler
import geopandas as gpd
import pickle

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
FIGURE_PATH = "../Figures/"
DATA_PATH = "../Data/"
DATA_LOAD_PATH = DATA_PATH + "original_data/"
DATA_SAVE_PATH = DATA_PATH + "processed_data/"

In [ ]:
from style_jcp import load_world
world = load_world()

In [ ]:
class Preprocess:
    def __init__(self, df, time_label=None):
        self.df = df.copy()
        self.time = time_label
        self.normalization_params = {}

    def preprocess_empirical_dataset(self):
        self.df.rename(columns={"Longitude": "lon", "Latitude": "lat"}, inplace=True)
        self.df["dep"] = self.df["MAR"].values.astype(float)
        self.df.drop(["MAR", "LogMAR"], axis=1, inplace=True)
        self.df["dep"] = self.df["dep"].values * 1000 * 365 * 24 * 60 * 60
        self.df = self.df[(self.df["lat"] > -90) & (self.df["lat"] < 90) &
                          (self.df["lon"] > -180) & (self.df["lon"] <= 180)]
        self.df['log_dep'] = np.log10(self.df['dep'])

    def preprocess_simulated_dataset(self):
        self.df.reset_index(inplace=True)
        if self.time.lower() == "holocene":
            self.df = self.df[np.isclose(self.df['time'], 0.1)]
        elif self.time.lower() == "lgm":
            self.df = self.df[np.isclose(self.df['time'], 21)]
        self.df.drop(["time", "load"], axis=1, inplace=True)
        self.df.loc[self.df['lon'] > 180, 'lon'] = self.df['lon'] - 360

        self.df['log_dep'] = np.log10(self.df['dep'])

    def normalize_data(self, verbose=True):
        global_mean = float(self.df['log_dep'].mean())
        global_std = float(self.df['log_dep'].std())

        self.df['log_dep_norm_temp'] = (self.df['log_dep'] - global_mean) / global_std

        lat_bands = [-90, -30, 30, 90]
        self.df['lat_band'] = pd.cut(self.df['lat'], bins=lat_bands, include_lowest=True)

        self.df['log_dep_norm'] = self.df['log_dep_norm_temp'].copy()
        bias_corrections = {}

        for band in self.df['lat_band'].unique():
            if pd.isna(band):
                continue

            mask = self.df['lat_band'] == band
            values = self.df.loc[mask, 'log_dep_norm_temp'].values
            local_mean = float(values.mean())

            bias_threshold = 0.2
            if abs(local_mean) > bias_threshold:
                correction = local_mean * 0.5
                self.df.loc[mask, 'log_dep_norm'] -= correction
                bias_corrections[str(band)] = {
                    'original_bias': local_mean,
                    'correction': correction,
                    'n_samples': int(len(values))
                }
                if verbose:
                    print(f"  {band}: 检测到偏差 {local_mean:.3f}，修正 {correction:.3f}")

        self.normalization_params = {
            'method': 'hybrid',
            'global_mean': global_mean,
            'global_std': global_std,
            'lat_bands': lat_bands,
            'bias_corrections': bias_corrections
        }

        if verbose:
            print(f"混合归一化: global_mean={global_mean:.4f}, global_std={global_std:.4f}")
            if not bias_corrections:
                print("  未检测到显著系统偏差，使用纯全局归一化")
        self.df.drop(['log_dep_norm_temp', 'lat_band'], axis=1, inplace=True)

In [ ]:
with open("functions_plot.py", 'r') as file:
    content = file.read()

# Execute the content of the .py file
exec(content)

In [ ]:
df_empirical_Holocene_raw = pd.read_csv(DATA_LOAD_PATH+"MAR_HOL.csv")
df_empirical_LGM_raw = pd.read_csv(DATA_LOAD_PATH+"MAR_LGM.csv")
df_simulation_raw = xr.open_dataset(DATA_LOAD_PATH+"Albanietal2016_GRL_dust.nc").to_dataframe()

In [ ]:
preprocessor_empirical_Holocene = Preprocess(df_empirical_Holocene_raw)
preprocessor_empirical_Holocene.preprocess_empirical_dataset()
preprocessor_empirical_Holocene.normalize_data()
df_empirical_Holocene = preprocessor_empirical_Holocene.df

preprocessor_empirical_LGM = Preprocess(df_empirical_LGM_raw)
preprocessor_empirical_LGM.preprocess_empirical_dataset()
preprocessor_empirical_LGM.normalize_data()
df_empirical_LGM = preprocessor_empirical_LGM.df

preprocessor_simulation_Holocene = Preprocess(df_simulation_raw, "Holocene")
preprocessor_simulation_Holocene.preprocess_simulated_dataset()
preprocessor_simulation_Holocene.normalize_data()
df_simulation_Holocene = preprocessor_simulation_Holocene.df

preprocessor_simulation_LGM = Preprocess(df_simulation_raw, "LGM")
preprocessor_simulation_LGM.preprocess_simulated_dataset()
preprocessor_simulation_LGM.normalize_data()
df_simulation_LGM = preprocessor_simulation_LGM.df

In [ ]:
from functions_plot import plot_dust_deposition_map

plot_dust_deposition_map(df=df_empirical_Holocene,
                         title='Holocene',
                         name_to_save='DATA_MAP_EMPIRICAL_HOLOCENE')

plot_dust_deposition_map(df=df_simulation_Holocene,
                         title='Holocene',
                         name_to_save='DATA_MAP_SIMULATED_HOLOCENE')

plot_dust_deposition_map(df=df_empirical_LGM,
                         title='Last Glacial Maximum',
                         name_to_save='DATA_MAP_EMPIRICAL_LGM')

plot_dust_deposition_map(df=df_simulation_LGM,
                         title='Last Glacial Maximum',
                         name_to_save='DATA_MAP_SIMULATED_LGM')

In [ ]:
from functions_plot import plot_hist

plot_hist(df=df_empirical_Holocene,
          title='Empirical data for Holocene',
          name_to_save='DATA_HIST_EMPIRICAL_HOLOCENE')

plot_hist(df=df_empirical_LGM,
          title='Empirical data for LGM',
          name_to_save='DATA_HIST_EMPIRICAL_LGM')

In [ ]:
with open(DATA_SAVE_PATH + "norm_params_empirical_Holocene.pkl", 'wb') as f:
    pickle.dump(preprocessor_empirical_Holocene.normalization_params, f)

with open(DATA_SAVE_PATH + "norm_params_empirical_LGM.pkl", 'wb') as f:
    pickle.dump(preprocessor_empirical_LGM.normalization_params, f)

with open(DATA_SAVE_PATH + "norm_params_simulation_Holocene.pkl", 'wb') as f:
    pickle.dump(preprocessor_simulation_Holocene.normalization_params, f)

with open(DATA_SAVE_PATH + "norm_params_simulation_LGM.pkl", 'wb') as f:
    pickle.dump(preprocessor_simulation_LGM.normalization_params, f)

In [ ]:
with open(DATA_SAVE_PATH + "df_empirical_Holocene.csv", 'w') as f:
    df_empirical_Holocene.to_csv(f, index=False)

with open(DATA_SAVE_PATH + "df_empirical_LGM.csv", 'w') as f:
    df_empirical_LGM.to_csv(f, index=False)

with open(DATA_SAVE_PATH + "df_simulation_Holocene.csv", 'w') as f:
    df_simulation_Holocene.to_csv(f, index=False)

with open(DATA_SAVE_PATH + "df_simulation_LGM.csv", 'w') as f:
    df_simulation_LGM.to_csv(f, index=False)